# CrewAI Samples

Willkommen! In diesem Notebook kannst du alle CrewAI-Beispiele direkt ausführen.

**So benutzt du dieses Notebook:**
- Klicke auf das ▶ Symbol neben eine Zelle und drücke **Ctrl+Alt+Enter** um sie auszuführen
- Ihr könnt den Text innerhalb der Code Zellen ändern und sie immer wieder ausführen

---

## Setup

Diese Zelle muss **einmalig am Anfang** ausgeführt werden.
Sie lädt den API-Key und prüft ob alles funktioniert.

In [ ]:
import os
from dotenv import load_dotenv

# Lädt OPENAI_API_KEY aus der .env Datei
load_dotenv()

if os.getenv("OPENAI_API_KEY"):
    print("✓ API-Key gefunden - bereit!")
else:
    print("✗ Kein API-Key gefunden! Bitte .env Datei prüfen.")

---
## Sample 1 - Ein einzelner Agent

Das einfachste CrewAI-Programm: **1 Agent, 1 Aufgabe, 1 Crew.**

Ein **Agent** ist wie eine KI-Person mit:
- einer **Rolle** (z.B. "Tier-Experte")
- einem **Ziel** (was will er erreichen?)
- einer **Hintergrundgeschichte** (das macht die KI besser in ihrer Rolle)

Eine **Aufgabe** beschreibt:
- was der Agent genau tun soll (`description`)
- wie das Ergebnis aussehen soll (`expected_output`)

### Deine Aufgabe:
Ersetze alle **TODO**-Texte im Code unten durch eigene Beschreibungen.
Denk dir eine kreative Rolle für deinen Agenten aus – es muss kein Tier-Experte sein!

**Beispiele für andere Themen:**
- Musikexpertin die Songempfehlungen gibt
- Reiseführerin die Städte beschreibt
- Köchin die Rezepte erfindet

> **Tipp:** Je genauer du `role`, `goal` und `backstory` beschreibst, desto besser wird die Antwort der KI!


In [ ]:
from crewai import Agent, Task, Crew
from IPython.display import display, Markdown

# Mit input kannst du die Eingabe dynamisch gestalten, also bei der Auführung Information von der Nutzerin abfragen.

# ── Hier wird beispielsweise ein Tier abgefragt, wenn nichts eingegeben wird, wird "Axolotl" verwendet ─────────────────────
TIER = input("Welches Tier soll beschrieben werden? (z.B. Axolotl, Drache, Krake): ").strip() or "Axolotl"
# ─────────────────────────────────────────────────────────

# Schritt 1: Agent erstellen
tier_experte = Agent(
    role="TODO: Kurzer Rollentitel für euren Agenten, z.B. Tier-Experte und Naturforscher",
    goal="TODO: Beschreibe das Ziel des Agenten, z.B. Erstelle faszinierende und lehrreiche Steckbriefe ueber Tiere.",
    backstory="""\
        TODO: Beschreibe die Hintergrundgeschichte des Agenten, z.B. wie er zum Tier-Experten wurde, welche Abenteuer er erlebt hat, etc.
    """,
    verbose=False,
)

# Schritt 2: Aufgabe erstellen
aufgabe = Task(
    description=f"""\
        TODO: Beschreibe die Aufgabe, auf die Eingabe kannst du wie folgt zugreifen: {TIER}
    """,
    expected_output="""\
        TODO: Beschreibe das erwartete Ergebnis
    """,
    agent=tier_experte,
)

# Schritt 3: Crew starten
crew = Crew(
    agents=[tier_experte],
    tasks=[aufgabe],
    verbose=False,
)

ergebnis = crew.kickoff()

display(Markdown(f"""
---
## Steckbrief: {TIER}

{ergebnis.raw}
"""))

---
## Sample 2 – Recherche-Agent

In diesem Sample baust du einen Agenten der aktuelle Informationen recherchiert.
Wir machen das in drei Teilen – jeder Teil ist eine kleine Erweiterung des vorherigen.

### Deine Aufgabe:
Ersetze in jedem Teil die **TODO**-Texte durch eigene Beschreibungen.
Das Thema (Wahlumfragen) ist vorgegeben – so kannst du die Ergebnisse gut vergleichen.


In [ ]:
from crewai import Agent, Task, Crew
from IPython.display import display, Markdown

# Agent OHNE Tool - nur Trainingswissen
# Kein tools=[] --> kein Internetzugang!
agent_ohne_tool = Agent(
    role="TODO: Kurzer Rollentitel, z.B. Politischer Analyst",
    goal="TODO: Was soll der Agent erreichen?",
    backstory="""        
        TODO: Hintergrundgeschichte des Agenten
    """,
    verbose=False,
)

aufgabe_ohne_tool = Task(
    description="""        
        TODO: Was soll der Agent herausfinden?
        Tipp: Frage nach aktuellen Wahlumfragen fuer Deutschland mit konkreten Prozentzahlen.
    """,
    expected_output="TODO: Wie soll das Ergebnis aussehen?",
    agent=agent_ohne_tool,
)

ergebnis_ohne = Crew(agents=[agent_ohne_tool], tasks=[aufgabe_ohne_tool]).kickoff()

display(Markdown(f"""
---
## Ergebnis: Ohne Web-Tool

{ergebnis_ohne.raw}
"""))


---
### Teil B – Agent mit Web-Tool

Jetzt bekommt der Agent ein Google-Such-Tool: `tools=[such_tool]`

Das ist der **einzige Unterschied** zum Code in Teil A.

**Voraussetzung:** `SERPER_API_KEY` in der `.env` Datei.  
Kostenlos registrieren (2500 Suchen gratis): [serper.dev](https://serper.dev)

Kopiere deine Beschreibungen aus Teil A – ändere nur den `tools`-Parameter.


In [ ]:
from crewai import Agent, Task, Crew
from crewai_tools import SerperDevTool
from IPython.display import display, Markdown

# Das Web-Such-Tool - liest SERPER_API_KEY automatisch aus .env
# Eine Übersicht über weitere Tools findet ihr hier: https://docs.crewai.com/en/concepts/tools
such_tool = SerperDevTool()

# Gleicher Agent wie in Teil A – nur tools=[such_tool] kommt dazu!
agent_mit_tool = Agent(
    role="TODO: Gleiche Rolle wie in Teil A",
    goal="TODO: Gleiches Ziel wie in Teil A",
    backstory="""        
        TODO: Gleiche Hintergrundgeschichte wie in Teil A
    """,
    verbose=False,
    tools=[such_tool],   # <-- das ist der einzige Unterschied zu Teil A!
)

aufgabe_mit_tool = Task(
    description="""        
        TODO: Gleiche Aufgabe wie in Teil A
    """,
    expected_output="TODO: Gleiches erwartetes Ergebnis wie in Teil A",
    agent=agent_mit_tool,
)

ergebnis_mit = Crew(agents=[agent_mit_tool], tasks=[aufgabe_mit_tool]).kickoff()

display(Markdown(f"""
---
## Ergebnis: Mit Web-Tool

{ergebnis_mit.raw}
"""))


---
### Teil C – Agent mit Web-Tool + heutigem Datum

Schau dir das Ergebnis aus Teil B genau an: Weiss der Agent welches Datum heute ist?

Ergänze im Task ganz oben eine einzige Zeile:
```python
from datetime import date
HEUTE = date.today().strftime("%d.%m.%Y")

description=f"""
    Heute ist der {HEUTE}.
    ...
"""
```

Kopiere deine Beschreibungen aus Teil B und füge diese Zeile hinzu.


In [ ]:
from datetime import date
from crewai import Agent, Task, Crew
from crewai_tools import SerperDevTool
from IPython.display import display, Markdown

# Heutiges Datum automatisch ermitteln
HEUTE = date.today().strftime("%d.%m.%Y")
print(f"Heutiges Datum wird übergeben: {HEUTE}")

such_tool = SerperDevTool()

agent_mit_datum = Agent(
    role="TODO: Gleiche Rolle wie in Teil B",
    goal="TODO: Gleiches Ziel wie in Teil B",
    backstory="""        
        TODO: Gleiche Hintergrundgeschichte wie in Teil B
    """,
    verbose=False,
    tools=[such_tool],
)

# <-- das ist der einzige Unterschied zu Teil B: die erste Zeile mit dem Datum!
aufgabe_mit_datum = Task(
    description=f"""        
        Heute ist der {HEUTE}.

        TODO: Gleiche Aufgabe wie in Teil B
    """,
    expected_output="TODO: Gleiches erwartetes Ergebnis wie in Teil B",
    agent=agent_mit_datum,
)

ergebnis_datum = Crew(agents=[agent_mit_datum], tasks=[aufgabe_mit_datum]).kickoff()

display(Markdown(f"""
---
## Ergebnis: Mit Web-Tool + Datum ({HEUTE})

{ergebnis_datum.raw}
"""))


### 💡 Was haben wir gelernt?

| | Teil A: Ohne Tool | Teil B: Mit Tool | Teil C: Mit Tool + Datum |
|---|---|---|---|
| **Datenquelle** | Trainingsdaten (veraltet) | Google-Suche | Google-Suche |
| **Datum bekannt?** | Nein ❌ | Nein ❌ | Ja ✅ |
| **Genauigkeit** | Veraltet | Besser, aber evtl. falsches Jahr | Aktuell ✅ |
| **Code-Änderung** | – | `tools=[such_tool]` | + `f"Heute ist der {HEUTE}"` |

**Was steckt dahinter?**

- **Teil A:** KI-Modelle werden zu einem bestimmten Zeitpunkt trainiert – alles was danach passiert, wissen sie nicht. Ohne Tool kann der Agent nur auf sein Trainingswissen zurückgreifen.
- **Teil B:** Mit einem Web-Tool kann der Agent aktiv googeln und aktuelle Seiten lesen. Aber er weiß immer noch nicht welches Datum heute ist – er könnte nach "aktuellen Umfragen 2024" suchen, obwohl wir längst weiter sind.
- **Teil C:** Wir übergeben das heutige Datum einfach im Task-Text. KI-Agenten wissen nicht was **"heute"** ist – wir müssen es ihnen immer explizit sagen!


---
## Sample 3 – Agenten-Zusammenarbeit: Die Debatte

Bisher hat immer nur **ein Agent** eine Aufgabe erledigt.
Jetzt arbeiten **drei Agenten zusammen** – wie ein echtes Team.

### Neues Konzept: `context=[]`
Agenten können die Ergebnisse anderer Agenten als Eingabe bekommen:

```python
aufgabe_contra = Task(
    ...
    context=[aufgabe_pro]  # <-- liest die Pro-Argumente!
)
```

### Deine Aufgabe:
1. Wähle eine **These** (z.B. `"Hausaufgaben sollten abgeschafft werden"`)
2. Ersetze alle **TODO**-Texte – beschreibe die drei Agenten und ihre Aufgaben
3. Überlege: Welche Rolle hat jeder Agent? Was soll er tun?


In [ ]:
from crewai import Agent, Task, Crew, Process
from IPython.display import display, Markdown

# ── Hier kannst du die These ändern ──────────────────────
THESE = "TODO: Deine These, z.B. Jede Familie sollte mindestens ein Haustier haben."
# ─────────────────────────────────────────────────────────

pro_agent = Agent(
    role="TODO: Rolle des Pro-Agenten",
    goal=f"TODO: Ziel des Pro-Agenten (Tipp: bezieht sich auf die These: {THESE})",
    backstory="""        
        TODO: Hintergrundgeschichte des Pro-Agenten
    """,
    verbose=False,
)

contra_agent = Agent(
    role="TODO: Rolle des Contra-Agenten",
    goal=f"TODO: Ziel des Contra-Agenten",
    backstory="""        
        TODO: Hintergrundgeschichte des Contra-Agenten
    """,
    verbose=False,
)

richter_agent = Agent(
    role="TODO: Rolle des Richter-Agenten",
    goal="TODO: Ziel des Richter-Agenten",
    backstory="""        
        TODO: Hintergrundgeschichte des Richter-Agenten
    """,
    verbose=False,
)

aufgabe_pro = Task(
    description=f"""        
        TODO: Aufgabe fuer den Pro-Agenten.
        Die These lautet: {THESE}
    """,
    expected_output="TODO: Wie soll das Ergebnis des Pro-Agenten aussehen?",
    agent=pro_agent,
)

aufgabe_contra = Task(
    description=f"""        
        TODO: Aufgabe fuer den Contra-Agenten.
        Die These lautet: {THESE}
    """,
    expected_output="TODO: Wie soll das Ergebnis des Contra-Agenten aussehen?",
    agent=contra_agent,
    context=[aufgabe_pro],  # <-- der Contra-Agent liest die Pro-Argumente!
)

aufgabe_urteil = Task(
    description=f"""        
        TODO: Aufgabe fuer den Richter.
        Die These lautet: {THESE}
    """,
    expected_output="TODO: Wie soll das Urteil aussehen?",
    agent=richter_agent,
    context=[aufgabe_pro, aufgabe_contra],  # <-- der Richter liest beide Seiten!
)

crew = Crew(
    agents=[pro_agent, contra_agent, richter_agent],
    tasks=[aufgabe_pro, aufgabe_contra, aufgabe_urteil],
    process=Process.sequential,
    verbose=False,
)

ergebnis = crew.kickoff()

display(Markdown(f"""
---
## Debatte: {THESE}

### Pro-Argumente
{aufgabe_pro.output.raw}

---
### Contra-Argumente
{aufgabe_contra.output.raw}

---
### Urteil
{ergebnis.raw}
"""))


### 💡 Was haben wir gelernt?

- **`context=[...]`** gibt einem Agenten die Ergebnisse anderer Agenten als Eingabe
- **`Process.sequential`** stellt sicher dass die Agenten nacheinander arbeiten
- Jeder Agent hat eine klar andere **Perspektive** — obwohl es dieselbe KI ist!
- Das zeigt: KI hat keine echte Meinung, sie nimmt die Rolle an die wir ihr geben

---
## Sample 4 – KI entwickelt ein Spiel

Das komplexeste Beispiel: Mehrere KI-Crews arbeiten zusammen, um ein **Text-Adventure zu entwickeln** – von der Idee bis zum fertigen Python-Code.

### Der Prozess hat 3 Phasen:

| Phase | Wer | Was |
|-------|-----|-----|
| 1 💡 Idee | 1 Agent | Erfindet ein Spielkonzept zum Thema das du wählst |
| 2 🎮 Design | 3 Agenten | Erstellt Räume, Gegenstände, Rätsel, Enden und ASCII-Art |
| 3 👩‍💻 Code | 2 Agenten | Schreibt und reviewed den fertigen Python-Code |

Nach jeder Phase kannst du **Feedback geben** – die KI überarbeitet das Ergebnis so lange, bis du zufrieden bist!

> **Hinweis:** Dieses Sample läuft im Terminal unten (nicht im Notebook).
> Alle Eingaben machst du dort – einfach die Zelle ausführen und das Terminal beobachten!

### Was du lernst:
- Mehrere `Crew`s hintereinander schalten
- **Feedback-Schleife:** Crew läuft erneut bis das Ergebnis passt
- Ergebnisse einer Crew als Eingabe für die nächste nutzen
- KI-generierter Code wird als `.py` Datei gespeichert – mit ASCII-Art pro Raum!


In [ ]:
# ==============================================================================
# Sample 4: KI entwickelt ein Spiel
# ==============================================================================
# Dieses Sample braucht ein echtes Terminal fuer die Eingaben.
# Fuehre den Befehl unten im Terminal aus (Tab "Terminal" in VS Code).
# ==============================================================================

import sys
import os
from IPython.display import display, Markdown

script = os.path.join(os.getcwd(), "sample04_game_dev.py")
python = sys.executable  # derselbe Python-Interpreter der gerade laeuft

display(Markdown(f"""
## Sample 4 starten

Dieses Sample laeuft im Terminal, nicht im Notebook – so funktionieren die Eingaben zuverlaessig.

**So gehts:**
1. Oeffne das Terminal in VS Code (unten, Tab **Terminal**)
2. Tippe diesen Befehl ein und druecke Enter:

```
& "{python}" "{script}"
```

Alle Eingaben und Ergebnisse erscheinen direkt im Terminal.
"""))


---
## Sample 5 – Quiz-Spiel (Vollständiges Beispiel)

Das vollständige Quiz-Spiel aus `sample05_quiz_game.py` mit **3 Agenten** die zusammenarbeiten:

| Agent | Aufgabe |
|---|---|
| **Fragen-Ersteller** | Erfindet Multiple-Choice-Fragen zum Thema |
| **Quiz-Master** | Stellt die Fragen und wertet aus |
| **Erklärer** | Erklärt die richtigen Antworten am Ende |

In [ ]:
# Quiz-Spiel direkt aus sample05_quiz_game.py ausführen

import os, sys
sys.path.insert(0, os.getcwd())

import importlib
import sample05_quiz_game as quiz_game
importlib.reload(quiz_game)
quiz_game.main()